In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.notebook import tqdm

from rtnls_enface.bounds import make_roi_mask_from_bounds
from vascx.fundus.loader import RetinaLoader
from vascx.fundus.retina import Retina

%matplotlib inline

# Reconstructing partially visible optic discs

Normal VascX extraction now fits and validates the optic disc once during loading. This notebook displays that accepted `retina.disc.ellipse`, which supplies the center, area-equivalent circle, directional landmarks and disc–fovea distances used by measurements. The six **RAV** samples have partially visible discs; the six other samples act as complete-disc controls.

The anatomical contour excludes the cut edge near the CFI/image boundary. The center, outline and full reconstructed mask can extend beyond the image. The observed segmentation and its center of mass remain available; vessel exclusion still uses the observed mask.

A fit is rejected only if **both** `visible_iou < 0.80` and `contour_rmse_px > max(2.0, 0.05 * semi_minor_axis)` hold, or fitting fails. Loaders retain their catch-and-clear behavior (`disc=None`). Measurement QC checks the complete requested region against CFI/image bounds, without a blanket disc-margin requirement.

Short arcs do not uniquely determine the hidden ellipse. The existing diagnostics and sensitivity plots are available for inspection; visible fraction, short arcs and diagnostic flags are not additional rejection criteria and are not propagated as uncertainty.

In [ ]:
# Locate the VascX repository when running from its root or a notebook subfolder.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "samples" / "fundus").is_dir())
SAMPLES_DIR = REPO_ROOT / "samples" / "fundus"
OUTPUT_DIR = REPO_ROOT / "samples" / "disc_ellipse_fits"

# Exclude contour pixels this close to the visible boundary, in 1024-image pixels.
BOUNDARY_MARGIN_PX = 5.0
# Weak preference for roundness on censored contours; zero gives an unregularized fit.
ROUNDNESS_WEIGHT = 0.02
# Robust-loss scale for residual contour errors, in pixels.
ROBUST_SCALE_PX = 2.0
# Compare the extrapolation under different assumptions about roundness.
SENSITIVITY_WEIGHTS = (0.0, 0.02, 0.05)

In [ ]:
loader = RetinaLoader.from_folder(SAMPLES_DIR)
retinas = {}
fits = {}
rows = []

for item in tqdm(loader.to_dict(), desc="Fit sample optic discs"):
    kwargs = dict(item)
    # Metadata bounds describe the original acquisition; warp them to the sample canvas.
    kwargs["roi_mask"] = make_roi_mask_from_bounds(kwargs.pop("bounds"), target_diameter=1024)
    retina = Retina.from_file(**kwargs)
    if retina.disc is None:
        raise ValueError(f"No optic disc found for {item['id']}")
    # Use the accepted geometry already selected by normal VascX loading.
    fit = retina.disc.ellipse
    image_id = item["id"]
    retinas[image_id] = retina
    fits[image_id] = fit
    rows.append({
        "sample": image_id,
        "observed_center_x": retina.disc.center_of_mass.x,
        "observed_center_y": retina.disc.center_of_mass.y,
        "fitted_center_x": fit.center.x,
        "fitted_center_y": fit.center.y,
        "semi_major_px": fit.semi_axes[0],
        "semi_minor_px": fit.semi_axes[1],
        "angle_degrees": fit.angle_degrees,
        "equivalent_radius_px": fit.equivalent_radius,
        "disc_fovea_distance_px": retina.disc_fovea_distance,
        "observed_disc_fovea_distance_px": retina.disc.center_of_mass.distance_to(retina.fovea_location),
        "center_shift_px": fit.center.distance_to(retina.disc.center_of_mass),
        "estimated_visible_percent": 100 * fit.visible_fraction,
        "estimated_outside_percent": 100 * (1 - fit.visible_fraction),
        "visible_iou": fit.visible_iou,
        "contour_rmse_px": fit.contour_rmse_px,
        "observed_arc_degrees": fit.observed_arc_degrees,
        "retained_contour_points": len(fit.contour_xy),
        "excluded_contour_points": len(fit.excluded_contour_xy),
        "flags": ", ".join(fit.flags),
    })

summary = pd.DataFrame(rows).set_index("sample")
rav_ids = [name for name in retinas if name.startswith("RAV_")]
control_ids = [name for name in retinas if not name.startswith("RAV_")]
display(summary.round(3))

## Fitted outlines

**White:** observed segmentation boundary. **Green:** completed ellipse, fitted center and directional landmarks. **Cyan:** retained contour points. **Magenta:** excluded contour near the cut edge. **Yellow:** CFI boundary. **Orange cross:** center of mass of the observed, truncated mask. The grey canvas outside the source image allows the entire reconstructed ellipse to remain visible.

“Outside” is the estimated fraction of the fitted ellipse outside the CFI **or** the image, using the full reconstructed area as denominator. It is not the fraction of the observed mask that was excluded from fitting.

In [ ]:
def plot_fits(sample_ids, title):
    ncols = 3
    nrows = int(np.ceil(len(sample_ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.8 * nrows), squeeze=False)
    for ax, image_id in zip(axes.flat, sample_ids):
        retina, fit = retinas[image_id], fits[image_id]
        ax.set_facecolor("#30343b")
        ax.imshow(retina.image)
        ax.contour(retina.mask, levels=[0.5], colors=["yellow"], linewidths=1)
        ax.contour(retina.disc.mask, levels=[0.5], colors=["white"], linewidths=.8)
        xy = fit.outline_xy()
        ax.plot(xy[:, 0], xy[:, 1], color="lime", lw=1.8)
        ax.scatter(*fit.contour_xy.T, s=3, c="cyan", zorder=3)
        if len(fit.excluded_contour_xy):
            ax.scatter(*fit.excluded_contour_xy.T, s=5, c="magenta", zorder=3)
        ax.plot(*retina.disc.center_of_mass.tuple_xy, marker="x", color="orange", ms=9)
        ax.plot(*fit.center.tuple_xy, marker="+", color="lime", ms=11)
        for point in (retina.disc.temporal_point, retina.disc.nasal_point,
                      retina.disc.superior_point, retina.disc.inferior_point):
            ax.plot(*point.tuple_xy, marker="o", color="lime", ms=3)
        x0, x1, y0, y1 = fit.bounding_box
        bx0, bx1, by0, by1 = retina.disc.bounding_box
        pad = max(x1 - x0, y1 - y0) * 0.12
        ax.set_xlim(min(x0, bx0) - pad, max(x1, bx1) + pad)
        ax.set_ylim(max(y1, by1) + pad, min(y0, by0) - pad)
        ax.set_aspect("equal")
        flags = ", ".join(fit.flags) or "no fit flags"
        ax.set_title(f"{image_id} | estimated outside: {1-fit.visible_fraction:.1%}\n"
                     f"visible IoU {fit.visible_iou:.3f} | arc {fit.observed_arc_degrees:.0f}° | {flags}", fontsize=10)
        ax.set_xlabel("x (source pixels)")
        ax.set_ylabel("y (source pixels)")
    for ax in axes.flat[len(sample_ids):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    return fig

rav_figure = plot_fits(rav_ids, "RAV: reconstructed optic-disc ellipses")
plt.show()

In [ ]:
control_figure = plot_fits(control_ids, "Complete-disc controls")
plt.show()

## Sensitivity of the hidden portion

The solid green outline uses the default weak regularization. The other outlines use no roundness penalty and a stronger penalty. Large differences outside the field indicate that the visible arc is insufficient to determine a unique reconstruction. `limited_visible_arc` flags a fitted angular span below 180°; the other flags identify poor overlap, large contour residuals, or an optimizer reaching its extrapolation limits. These are exploratory diagnostics, not calibrated confidence bounds.

The boundary exclusion margin is also adjustable. The table below compares 3, 5, and 8 pixels under the default regularization.

In [ ]:
sensitivity = []
sensitivity_figure, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
colors = ("deepskyblue", "lime", "orange")
for ax, image_id in zip(axes.flat, rav_ids):
    retina = retinas[image_id]
    ax.set_facecolor("#30343b")
    ax.imshow(retina.image)
    ax.contour(retina.mask, levels=[0.5], colors=["yellow"], linewidths=1)
    ax.contour(retina.disc.mask, levels=[0.5], colors=["white"], linewidths=1)
    outlines = []
    for weight, color in zip(SENSITIVITY_WEIGHTS, colors):
        fit = retina.disc.fit_ellipse(boundary_margin_px=BOUNDARY_MARGIN_PX, roundness_weight=weight)
        xy = fit.outline_xy()
        outlines.append(xy)
        ax.plot(*xy.T, color=color, label=f"roundness={weight:g}")
        sensitivity.append({"sample": image_id, "roundness_weight": weight,
                            "boundary_margin_px": BOUNDARY_MARGIN_PX,
                            "center_x": fit.center.x, "center_y": fit.center.y,
                            "estimated_outside_percent": 100*(1-fit.visible_fraction),
                            "visible_iou": fit.visible_iou, "flags": ", ".join(fit.flags)})
    xy = np.concatenate(outlines)
    lo, hi = xy.min(axis=0), xy.max(axis=0)
    pad = max(hi-lo)*0.08
    ax.set_xlim(lo[0]-pad, hi[0]+pad)
    ax.set_ylim(hi[1]+pad, lo[1]-pad)
    ax.set_aspect("equal")
    ax.set_title(image_id)
    ax.legend(fontsize=8)
sensitivity_figure.suptitle("How much does the hidden ellipse depend on the roundness assumption?")
sensitivity_figure.tight_layout()
plt.show()

for image_id in tqdm(rav_ids, desc="Boundary-margin sensitivity"):
    for margin in (3.0, 8.0):
        fit = retinas[image_id].disc.fit_ellipse(boundary_margin_px=margin, roundness_weight=ROUNDNESS_WEIGHT)
        sensitivity.append({"sample": image_id, "roundness_weight": ROUNDNESS_WEIGHT,
                            "boundary_margin_px": margin,
                            "center_x": fit.center.x, "center_y": fit.center.y,
                            "estimated_outside_percent": 100*(1-fit.visible_fraction),
                            "visible_iou": fit.visible_iou, "flags": ", ".join(fit.flags)})
sensitivity_table = pd.DataFrame(sensitivity)
display(sensitivity_table.round(3))

## Export the accepted geometry

`fit.full_mask()` returns a complete binary mask and its `(row, column)` origin in source-image coordinates. The origin can be negative: preserve it when comparing the mask with CFI bounds. `fit.to_mask(retina.disc.mask.shape)` instead returns a source-sized mask and necessarily clips off-image pixels.

The exports retain the fit parameters and diagnostics. No existing sample mask, landmark, or regression reference is overwritten.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_DIR / "fit_summary.csv")
sensitivity_table.to_csv(OUTPUT_DIR / "fit_sensitivity.csv", index=False)
rav_figure.savefig(OUTPUT_DIR / "rav_ellipse_fits.png", dpi=160, bbox_inches="tight")
control_figure.savefig(OUTPUT_DIR / "control_ellipse_fits.png", dpi=160, bbox_inches="tight")
sensitivity_figure.savefig(OUTPUT_DIR / "fit_sensitivity.png", dpi=160, bbox_inches="tight")
for image_id, fit in fits.items():
    full_mask, origin_yx = fit.full_mask()
    np.savez_compressed(
        OUTPUT_DIR / f"{image_id}_ellipse.npz",
        mask=full_mask, origin_yx=origin_yx, center_yx=fit.center.tuple,
        semi_axes=fit.semi_axes, angle_degrees=fit.angle_degrees,
        boundary_margin_px=fit.boundary_margin_px, roundness_weight=fit.roundness_weight,
    )
print(f"Saved fit summaries, figures, and complete masks to {OUTPUT_DIR}")